In [79]:
import pandas_datareader.data as reader
import yfinance as yf
import pandas as pd
import datetime as dt
import statsmodels.api as sm

In [80]:
end = dt.date(2024,12,1)
start = dt.date(2014,1,1)

In [81]:
# factors= reader.DataReader('F-F_Research_Data_5_Factors_2x3_daily.CSV', 'famafrench', start, end)[0]
factors = reader.DataReader('F-F_Research_Data_5_Factors_2x3', 'famafrench', start, end)[0]

<ipython-input-81-0ca25a988cba>:2: FutureWarning: The argument 'date_parser' is deprecated and will be removed in a future version. Please use 'date_format' instead, or read your data in as 'object' dtype and then call 'to_datetime'.
  factors = reader.DataReader('F-F_Research_Data_5_Factors_2x3', 'famafrench', start, end)[0]
<ipython-input-81-0ca25a988cba>:2: FutureWarning: The argument 'date_parser' is deprecated and will be removed in a future version. Please use 'date_format' instead, or read your data in as 'object' dtype and then call 'to_datetime'.
  factors = reader.DataReader('F-F_Research_Data_5_Factors_2x3', 'famafrench', start, end)[0]


In [82]:
factors

,Mkt-RF,SMB,HML,RMW,CMA,RF
Date,,,,,,
2014-01,-3.32,0.58,-2.07,-3.87,-1.43,0.00
2014-02,4.65,0.13,-0.31,-0.21,-0.44,0.00
2014-03,0.43,-1.08,4.93,2.13,1.92,0.00
2014-04,-0.19,-4.11,1.17,3.47,1.00,0.00
2014-05,2.06,-1.89,-0.13,0.06,-1.00,0.00
...,...,...,...,...,...,...
2024-06,2.77,-4.37,-3.31,0.51,-1.78,0.41
2024-07,1.24,8.28,5.74,0.22,0.43,0.45
2024-08,1.61,-3.65,-1.13,0.85,0.86,0.48


In [83]:
factors.shape

(130, 6)

In [84]:
def handle_data(ticker):
  returns = yf.download(ticker,start,end)['Adj Close'].pct_change()
  returns_mtl = returns.resample('M').agg(lambda x: (x+1).prod()-1)[:-1]
  returns_mtl.index = factors.index
  return returns_mtl


In [85]:
handle_data('AAPL')

[*********************100%***********************]  1 of 1 completed
<ipython-input-84-ae754864078e>:3: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  returns_mtl = returns.resample('M').agg(lambda x: (x+1).prod()-1)[:-1]


Ticker,AAPL
Date,
2014-01,-0.094969
2014-02,0.057511
2014-03,0.019953
2014-04,0.099396
2014-05,0.078709
...,...
2024-06,0.095553
2024-07,0.054411
2024-08,0.032353


In [98]:
def fama(ticker):
  data = handle_data(ticker)
  merge = pd.merge(data, factors, on = 'Date')
  merge[['Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA']] = merge[['Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA']]/100
  merge['AAPL_excess'] = merge['AAPL'] - merge['RF']
  y = merge['AAPL_excess']
  X = merge[['Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA']]
  X = sm.add_constant(X)
  model = sm.OLS(y,X).fit()
  return model.summary()


In [100]:
tickers = ['AAPL','TSLA','QBTS', 'SLNO', '^GSPC','^DJI']
for ticker in tickers:
  print(ticker)
  print(fama(ticker))
  print('\n')

[*********************100%***********************]  1 of 1 completed

AAPL
                            OLS Regression Results                            
Dep. Variable:            AAPL_excess   R-squared:                       0.118
Model:                            OLS   Adj. R-squared:                  0.082
Method:                 Least Squares   F-statistic:                     3.311
Date:                Fri, 27 Dec 2024   Prob (F-statistic):            0.00767
Time:                        04:20:18   Log-Likelihood:                 51.240
No. Observations:                 130   AIC:                            -90.48
Df Residuals:                     124   BIC:                            -73.27
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.1145      0.015     -7.492    


<ipython-input-84-ae754864078e>:3: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  returns_mtl = returns.resample('M').agg(lambda x: (x+1).prod()-1)[:-1]
[*********************100%***********************]  1 of 1 completed
<ipython-input-84-ae754864078e>:3: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  returns_mtl = returns.resample('M').agg(lambda x: (x+1).prod()-1)[:-1]


KeyError: 'AAPL'